# Maritime Survival Detection Classification

## Overview
This notebook tackles the **Maritime Survival Detection** classification problem. Our goal is to predict the `Outcome` (survival) of passengers based on various features such as Age, Class, Gender, Deck, and Boarding Port.

The workflow includes:
1. **Data Loading & EDA**: Understanding the dataset and missing values.
2. **Feature Engineering**: Imputing missing data, creating new features (AgeGroup, TicketGroupSize, Deck), and encoding categorical variables.
3. **Model Selection**: Training baseline models like RandomForest, Logistic Regression, and LightGBM.
4. **Evaluation**: Using Stratified K-Fold Cross Validation.
5. **Prediction**: Generating final predictions on the test set.

In [1]:
# Import essential analytics libraries
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/survival-detection/maritime_train.csv
/kaggle/input/survival-detection/maritime_sample_submission.csv
/kaggle/input/survival-detection/maritime_test.csv


## 1. Data Loading and Initial Exploration

We begin by loading the `train` and `test` datasets from the provided CSV files and inspecting their structure.

In [2]:
# Load training and testing datasets
train=pd.read_csv('/kaggle/input/survival-detection/maritime_train.csv')
test=pd.read_csv('/kaggle/input/survival-detection/maritime_test.csv')

Let's preview the top rows of our training data:

In [3]:
train

,PassengerId,TicketTier,PassengerName,Gender,Age,RelativesAboard,ParentsChildren,CLass,TicketCost,Berth,BoardingPort,FamilySize,Singleton,FarePerPerson,Title,Outcome
0,338.376107,0.977533,"Partner, Mr. Austen",male,NaN,0.005991,-0.024062,113043,29.104891,C124,S,1,1,25.906873,Mr,0
1,732.225160,2.016456,"Berriman, Mr. William John",male,21.574386,0.091114,-0.021340,28425,11.033567,NaN,S,1,1,10.939094,Mr,0
2,391.314100,2.998448,"Tikkanen, Mr. Juho",male,33.490756,-0.072885,-0.021702,STON/O 2. 3101293,2.234540,NaN,S,1,1,4.065968,Mr,0
3,724.550481,3.045488,"Hansen, Mr. Henrik Juul",male,25.200171,0.913680,0.032986,350025,10.958320,NaN,S,2,0,3.733784,Mr,0
4,810.994274,3.004710,"Andersson, Miss. Ebba Iris Alfrida",female,5.839590,4.009691,1.956266,347082,33.765343,NaN,S,7,0,3.325612,Miss,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
707,129.525540,3.020938,"Salkjelsvik, Miss. Anna Kristine",female,21.014222,-0.068689,-0.001278,343120,2.464658,NaN,S,1,1,3.411855,Miss,1
708,244.275165,1.022194,"Cairns, Mr. Alexander",male,NaN,0.012751,0.025355,113798,28.086475,NaN,S,1,1,28.713427,Mr,0
709,882.776683,3.044219,"Hansen, Mr. Claus Peter",male,41.429067,1.935456,0.004872,350026,16.523350,NaN,S,3,0,6.395947,Mr,0
710,438.708741,0.984953,"Carter, Miss. Lucile Polk",female,NaN,0.965363,1.995522,113760,119.656646,B96 B98,S,4,0,30.547369,Miss,1


And the test data:

In [4]:
test

,PassengerId,TicketTier,PassengerName,Gender,Age,RelativesAboard,ParentsChildren,CLass,TicketCost,Berth,BoardingPort,FamilySize,Singleton,FarePerPerson,Title
0,717.352816,3.000703,"Moubarek, Master. Halim Gonios (""William George"")",male,NaN,0.975830,0.998900,2661,15.879511,NaN,C,3,0,8.320836,Master
1,461.293943,2.034863,"Kvillner, Mr. Johan Henrik Johannesson",male,30.588645,0.027526,-0.009372,C.A. 18723,5.759037,NaN,S,1,1,11.233223,Mr
2,838.122587,3.042718,"Alhomaki, Mr. Ilmari Rudolf",male,19.758546,-0.022452,0.002519,SOTON/O2 3101287,10.671461,NaN,S,1,1,7.985473,Mr
3,721.903019,1.977906,"Harper, Miss. Annie Jessie ""Nina""",female,7.227357,-0.031287,0.973938,248727,34.326022,NaN,S,2,0,17.385664,Miss
4,42.507124,2.993392,"Nicola-Yarred, Miss. Jamila",female,13.798622,0.977654,-0.014813,2651,8.282181,NaN,C,2,0,5.173044,Miss
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174,431.602888,2.964833,"Kallio, Mr. Nikolai Erland",male,17.369764,0.019143,0.054858,STON/O 2. 3101274,8.810211,NaN,S,1,1,6.899390,Mr
175,759.287936,2.984767,"Elias, Mr. Dibo",male,NaN,-0.014610,-0.042299,2674,7.898009,NaN,C,1,1,4.175122,Mr
176,29.553908,3.033819,"Asplund, Mrs. Carl Oscar (Selma Augusta Emilia...",female,37.091634,0.972775,4.930177,347077,34.346621,NaN,S,7,0,4.212042,Mrs
177,57.993443,1.953763,"Ilett, Miss. Bertha",female,15.898678,-0.003835,0.016434,SO/C 14885,10.646962,NaN,S,1,1,11.688372,Miss


### Dataset Information
Checking feature data types and non-null counts.

In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 712 entries, 0 to 711
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   PassengerId      712 non-null    float64
 1   TicketTier       712 non-null    float64
 2   PassengerName    712 non-null    object 
 3   Gender           712 non-null    object 
 4   Age              515 non-null    float64
 5   RelativesAboard  712 non-null    float64
 6   ParentsChildren  712 non-null    float64
 7   CLass            712 non-null    object 
 8   TicketCost       712 non-null    float64
 9   Berth            159 non-null    object 
 10  BoardingPort     710 non-null    object 
 11  FamilySize       712 non-null    int64  
 12  Singleton        712 non-null    int64  
 13  FarePerPerson    712 non-null    float64
 14  Title            712 non-null    object 
 15  Outcome          712 non-null    int64  
dtypes: float64(7), int64(3), object(6)
memory usage: 89.1+ KB


Checking for missing values across all columns:

In [6]:
train.isna().sum()

PassengerId          0
TicketTier           0
PassengerName        0
Gender               0
Age                197
RelativesAboard      0
ParentsChildren      0
CLass                0
TicketCost           0
Berth              553
BoardingPort         2
FamilySize           0
Singleton            0
FarePerPerson        0
Title                0
Outcome              0
dtype: int64

## 2. Feature Engineering and Data Preprocessing

Based on our EDA, we have missing values and categorical features that need to be processed before feeding them to a machine learning model.

### Imputing Missing Values
We'll use a `SimpleImputer` with the `median` strategy to fill missing numerical values in the `Age` column, which is robust to outliers.

In [7]:
from sklearn.impute import SimpleImputer

# Instantiate imputer and fill missing Age values with the median
imputer = SimpleImputer(strategy='median')
train['Age'] = imputer.fit_transform(train[['Age']])
test['Age'] = imputer.transform(test[['Age']]) # Ensure we use the same imputer on test data

### Feature Creation: Ticket Group Size
We generate a new feature mapping the `CLass` to the frequency of tickets, capturing group sizes.

In [9]:
# Map class frequency to create a TicketGroupSize feature
ticket_counts = train['CLass'].value_counts() 
train['TicketGroupSize'] = train['CLass'].map(ticket_counts)

ticket_counts_test = test['CLass'].value_counts() 
test['TicketGroupSize'] = test['CLass'].map(ticket_counts_test)

### Feature Creation: Age Groups
Binning continuous `Age` into categorical bins (`Child`, `Teen`, `Adult`, `MiddleAged`, `Senior`) can help the model capture non-linear relationships.

In [10]:
# Define age bins and labels
bins = [0, 12, 18, 40, 60, 120]
labels = ['Child', 'Teen', 'Adult', 'MiddleAged', 'Senior']

# Apply binning
train['AgeGroup'] = pd.cut(train['Age'], bins=bins, labels=labels)
test['AgeGroup'] = pd.cut(test['Age'], bins=bins, labels=labels)

### Feature Extraction: Deck from Berth
Extracting the deck level (first letter) from the `Berth` attribute.

In [11]:
# Extract the first letter of the Berth to represent the Deck level
train['Deck'] = train['Berth'].str[0].fillna('Unknown')
test['Deck'] = test['Berth'].str[0].fillna('Unknown')

### Categorical Encoding
Applying One-Hot Encoding (`get_dummies`) to our categorical features.

In [12]:
# List of categorical columns to encode
cat_cols = ['Gender', 'CLass', 'Berth', 'BoardingPort', 'Title', 'AgeGroup', 'Deck']

# Apply pandas get_dummies for One-Hot Encoding
train_encoded = pd.get_dummies(train, columns=cat_cols)
test_encoded = pd.get_dummies(test, columns=cat_cols)

### Feature Scaling
Scaling `PassengerId` and `Age` using `StandardScaler` to ensure numerical features possess a similar scale, which is crucial for distance-based and regularized models.

In [13]:
from sklearn.preprocessing import StandardScaler 

# Initialize scaler
scaler = StandardScaler()

# Scale PassengerId
train_encoded['PassengerId_scaled'] = scaler.fit_transform(train_encoded[['PassengerId']])
test_encoded['PassengerId_scaled'] = scaler.transform(test_encoded[['PassengerId']])

# Scale Age
train_encoded['Age_scaled'] = scaler.fit_transform(train_encoded[['Age']])
test_encoded['Age_scaled'] = scaler.transform(test_encoded[['Age']])

Let's inspect the columns after feature engineering:

In [14]:
train_encoded.columns

Index(['PassengerId', 'TicketTier', 'PassengerName', 'Age', 'RelativesAboard',
       'ParentsChildren', 'TicketCost', 'FamilySize', 'Singleton',
       'FarePerPerson',
       ...
       'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T',
       'Deck_Unknown', 'Deck_nan', 'PassengerId_scaled'],
      dtype='object', length=720)

### Preparing Training Data
We separate our target variable `Outcome` and drop unnecessary columns like `PassengerName`.

In [16]:
# Separate features (X) and target (y)
x = train_encoded.drop(['Outcome', 'PassengerName'], axis=1)
y = train_encoded['Outcome']

In [17]:
# Drop PassengerId as those shouldn't be predictive features
x = x.drop(columns=['PassengerId', 'PassengerId_scaled'], axis=1)

In [18]:
x.columns

Index(['TicketTier', 'Age', 'RelativesAboard', 'ParentsChildren', 'TicketCost',
       'FamilySize', 'Singleton', 'FarePerPerson', 'TicketGroupSize',
       'SharedTicket',
       ...
       'Deck_A', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G',
       'Deck_T', 'Deck_Unknown', 'Deck_nan'],
      dtype='object', length=716)

## 3. Model Training and Evaluation

We will evaluate several models to find the best performer on our dataset.

### Model 1: Random Forest Classifier
Building a Random Forest ensemble model with 350 estimators and a depth of 6.

In [19]:
from sklearn.ensemble import RandomForestClassifier

# Initialize Random Forest with tuned hyperparameters
rf_model = RandomForestClassifier(n_estimators=350, max_depth=6, random_state=42)

RandomForestClassifier(n_estimators=350, random_state=42)

Evaluating Random Forest via 8-fold Cross-Validation:

In [43]:
from sklearn.model_selection import cross_val_score

# Calculate CV accuracy scores
scores = cross_val_score(rf_model, x, y, cv=8, scoring='accuracy')
print(f'Random Forest CV Accuracy: {scores.mean():.4f} +/- {scores.std():.4f}')

CV Accuracy: 0.8286516853932584


Alternatively, evaluating Random Forest with Stratified K-Fold manually:

In [44]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
accuracies = []

for train_idx, val_idx in skf.split(x, y):
    X_train_fold, X_val_fold = x.iloc[train_idx], x.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
    
    model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
    model.fit(X_train_fold, y_train_fold)
    
    preds = model.predict(X_val_fold)
    accuracies.append(accuracy_score(y_val_fold, preds))

print(f'Mean Stratified Accuracy: {sum(accuracies) / len(accuracies):.4f}')

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [07:07:53] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [07:07:54] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [07:07:54] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [07:07:56] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [07:07:57] WARNING: /w

CV Accuracy: [0.7916666666666666, 0.8055555555555556, 0.8873239436619719, 0.8591549295774648, 0.8450704225352113, 0.8169014084507042, 0.7464788732394366, 0.8169014084507042, 0.8450704225352113, 0.7605633802816901]
Mean Accuracy: 0.8174687010954618


### Model 2: Logistic Regression (Hyperparameter Tuning)
Using `RandomizedSearchCV` to find optimal `C` and `l1_ratio` values for Logistic Regression.

In [45]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
import numpy as np

# Define hyperparameter space
param_dist = {
    'C': np.logspace(-3, 3, 20),
    'l1_ratio': np.linspace(0, 1, 10)
}

# Initialize model and search
log_reg = LogisticRegression(penalty='elasticnet', solver='saga', max_iter=2000, random_state=42)
random_search = RandomizedSearchCV(log_reg, param_distributions=param_dist, n_iter=10, cv=5, scoring='accuracy', random_state=42, n_jobs=-1)

# Fit search
random_search.fit(x, y)
print('Best Logistic Regression params:', random_search.best_params_)
print('Best CV score:', random_search.best_score_)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which 

Best Params: {'solver': 'liblinear', 'penalty': 'l2', 'C': 10}
Best CV Accuracy: 0.8384831460674157


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


### Model 3: LightGBM
Applying a Gradient Boosting approach using LightGBM.

In [46]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Note: Clean column names beforehand if needed (LightGBM doesn't like special JSON characters)
import re
x = x.rename(columns = lambda col:re.sub('[^A-Za-z0-9_]+', '', col))

# Initialize and evaluate LightGBM
lgb_model = lgb.LGBMClassifier(n_estimators=100, random_state=42)
scores = cross_val_score(lgb_model, x, y, cv=5, scoring='accuracy')
print(f'LightGBM CV Accuracy: {scores.mean():.4f} +/- {scores.std():.4f}')

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 268, number of negative: 444
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003001 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1488
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.376404 -> initscore=-0.504838
[LightGBM] [Info] Start training from score -0.504838
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

## 4. Final Test Predictions

Finally, we prepare our test dataset identically to our training set, generate predictions using our best model, and format the output for submission.

In [51]:
# 1. Drop unused columns from the Test set
x_test = test_encoded.drop(['PassengerName'], axis=1)

# 2. Re-align columns to match the Training set exactly (adds missing encoded features with 0)
x_test = x_test.reindex(columns=x.columns, fill_value=0)

# Clean column names for LGBM as done with Training data
x_test = x_test.rename(columns = lambda col:re.sub('[^A-Za-z0-9_]+', '', col))

# 3. Fit our chosen model on the full training dataset
# (Using LightGBM as an example; swap to rf_model if preferred based on CV scores)
lgb_model.fit(x, y)

# 4. Generate Outcome predictions
test_predictions = lgb_model.predict(x_test)

# 5. Create final submission DataFrame
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Outcome': test_predictions
})

# Save to CSV
submission.to_csv('submission.csv', index=False)
print('Submission file "submission.csv" generated successfully!')
submission.head()